### 3장 1강: Retriever 구성과 질의 임베딩

**학습 목표**

- 사용자 질문을 임베딩 모델을 통해 고차원 벡터로 변환할 수 있다.
- PGVector 벡터 저장소 기반으로 검색기(Retriever) 인터페이스를 구성할 수 있다.
- 검색 파라미터를 설정하고 LLM 모델과 연동하여 문서 기반 질의응답을 검증할 수 있다.

In [4]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="integrated_university_rules",
    connection=connection_url,
    use_jsonb=True
)

initial_documents = [
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]

vector_store.add_documents(initial_documents)

['df45bde7-3cca-49f3-a231-8f160c92b68e',
 '1447dfdc-a043-4d45-a460-b7a833b65f70']

In [5]:
user_query = "학부생의 도서 대출 권수와 기간을 알려주세요."

retreiver = vector_store.as_retriever()

searched_docs = retreiver.invoke(user_query)

for doc in searched_docs:
    print(doc)

page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.' metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}
page_content='[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.' metadata={'year': 2026, 'source': 'rules_scholarship.txt', 'category': 'scholarship'}


In [6]:
# 단락 형태로 문장 만들기
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in searched_docs)


In [7]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    reasoning_effort="low"
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 대학 학사 규정 안내 전문가입니다.
    제공된 [참고 문서]에만 근거해서 답변하고, 
    문서에 없는 내용은 '죄송하지만 알지 못합니다.'라고 답변하세요.

    [참고 문서]
    {context}
    """),
    ("user", "{question}")
])

chain = (
    {"context": retreiver | format_docs, "question": RunnablePassthrough()}
     | prompt
     | model
     | parser # runnable - 최종 결과물을 context에 주입한다.
)

response = chain.invoke("도서관 대출 권수와 기간 규정을 알려줘.")
print(response)

학부생은 도서관에서 최대 5권까지 14일간 대출할 수 있습니다.


In [ ]:
# vectore_store.as_retriever() - 설정
"""
search_kwargs={
    "k": 3, Top-k, 유사도 기준 상위 3개 추출
    "filter": { .... }, metadata 필터링 조건 ...
    "score_threshold": ..., -> 필수로 search_type="similarity_score_threshold" 설정이 필수
}
"""

In [ ]:
query = "도서관 대출 권수와 기간 규정을 알려줘."
# similarity_search_with_score = 기준이 코사인 거리: 숫자가 작을수록 유사도가 높다
searched_docs1 = vector_store.similarity_search_with_score(
    query=query
)

for doc, score in searched_docs1:
    print(f"page_content={doc.page_content} | score={score:.4f}") 

page_content=[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다. | score=0.4494
page_content=[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다. | score=0.7657


In [16]:
# similarity_search_with_relevance_scores(..) : 유사도 임계치를 설정할 수 있는 메서드
SCORE_THRESHOLD = 0.7 # 최소 스코어 0.7 이상 -> 0.7 이상이 절대 안나온다.

# 스코어가 1에 가까울수록 유사 0~1
searched_docs2 = vector_store.similarity_search_with_relevance_scores(
    query=query,
    score_threshold= 1 - SCORE_THRESHOLD # data >= 0.7
)

for doc, relevance_score in searched_docs2:
    print(f"page_content={doc.page_content} | relevance score={relevance_score}")

page_content=[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다. | relevance score=0.5505663550565802


In [20]:
retreiver2 = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 1- SCORE_THRESHOLD}
)
searched_docs3 = retreiver2.invoke(query)

print(searched_docs3)

[Document(id='df45bde7-3cca-49f3-a231-8f160c92b68e', metadata={'year': 2026, 'source': 'rules_library.txt', 'category': 'library'}, page_content='[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.')]
